# Task 3 — Clean-Slate Screen 1

This notebook runs the first clean-slate model screen. It does not continue the old CNN chain. Gender and usage use separate model designs, and only canonical folds 0 and 4 are scored at this stage. The human observability review is deferred and does not block this run.


## 1. Local runtime paths

This experiment runs in the repository's `.venv`. It uses the local teacher dataset and saves generated artifacts under the gitignored `results/task3/` folder. No Colab session, Drive mount, or GPU is required.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src/fashion').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the MLA2 repository.')

REPO_DIR = find_repo_root()
LOCAL_TASK_DIR = REPO_DIR / 'results/task3'
LOCAL_REGISTRY = REPO_DIR / 'results/runs.csv'
LOCAL_FEATURE_WORK_DIR = REPO_DIR / 'tmp/task3-clean-slate-feature-work'
LOCAL_EVIDENCE_DIR = REPO_DIR / 'results/evidence/task3'


In [ ]:
expected_venv = (REPO_DIR / '.venv').resolve()
active_venv = Path(sys.prefix).resolve()
if active_venv != expected_venv:
    raise RuntimeError(
        f'Select the repository .venv kernel first: expected {expected_venv}, got {active_venv}'
    )

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
branch = subprocess.check_output(
    ['git', 'branch', '--show-current'], cwd=REPO_DIR, text=True
).strip()
print(f'Local repository: {REPO_DIR}')
print(f'Kernel environment: {active_venv}')
print(f'Branch: {branch}')
print(f'Commit: {commit}')
print(f'Artifact root: {LOCAL_TASK_DIR}')


In [ ]:
teacher_dir = REPO_DIR / 'data/raw/teacher'
required_files = (
    teacher_dir / 'train/styles_train.csv',
    teacher_dir / 'test/styles_prediction.csv',
)
image_dirs = (teacher_dir / 'train/images_train', teacher_dir / 'test/images_test')
image_suffixes = {'.jpg', '.jpeg'}

actual_images = sum(
    path.suffix.lower() in image_suffixes
    for image_dir in image_dirs
    for path in image_dir.glob('*')
)
missing_files = [str(path) for path in required_files if not path.is_file()]
missing_dirs = [str(path) for path in image_dirs if not path.is_dir()]
if missing_files or missing_dirs or actual_images == 0:
    raise RuntimeError(
        f'Local teacher data is incomplete: images={actual_images:,}, '
        f'missing_files={missing_files}, missing_dirs={missing_dirs}'
    )
print(f'Local teacher data ready: {actual_images:,} images')


## 2. Zero-fit preflight

This check reads the canonical split and class maps. It confirms the two distinct model contracts and the 7 GiB host-memory budget. It does not extract the full cache or fit a model.


In [ ]:
required_modules = ('numpy', 'pandas', 'PIL', 'skimage', 'sklearn')
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError(f'The local .venv is missing required packages: {missing_modules}')

os.chdir(REPO_DIR)
os.environ['FASHION_PROJECT_ROOT'] = str(REPO_DIR)
source_dir = str(REPO_DIR / 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
LOCAL_TASK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_REGISTRY.parent.mkdir(parents=True, exist_ok=True)

from fashion.train.task3_clean_slate import (
    check_clean_slate_screen_setup,
    prepare_clean_slate_screen_features,
    run_clean_slate_gender_screen,
    run_clean_slate_usage_screen,
)

preflight = check_clean_slate_screen_setup(root=REPO_DIR, folds=(0, 4))
if not preflight['training_screen_ready'] or preflight['training_blockers']:
    raise RuntimeError(f"Training is blocked: {preflight['training_blockers']}")
if preflight['model_fits'] != 0 or preflight['optimizer_steps'] != 0:
    raise RuntimeError('Preflight unexpectedly fitted a model.')
print('Human review:', preflight['human_observability_review_status'])
print('Gender model:', preflight['gender_model'])
print('Usage model:', preflight['usage_model'])
print('Screen folds:', preflight['folds'])
print('Estimated cache GiB:', round(preflight['estimated_two_view_cache_bytes'] / 1024**3, 3))


## 3. Frozen hypotheses

**Gender.** The old learned models nearly memorised their training rows. A fixed foreground-gradient representation has much lower freedom, while keeping the shape and colour signals supported by the EDA. A calibrated linear SVM may therefore reduce the train–validation gap without losing too much gender macro-F1.

**Usage.** Usage is strongly tied to product type, but the true article-type label is not available at Task 3 inference time. This model first predicts a full article-type probability distribution from the image. It then combines that uncertainty with a smoothed `P(usage | articleType)` table fitted only on the outer-fold training rows. This tests a task-shaped decision rule instead of another flat usage head.

These are independent clean-slate candidates. Neither is a child of E1–E10.


## 4. Build or reuse teacher-only feature caches

This is the longest preparation step. It hashes the teacher development images, then writes two label-blind feature matrices under `results/task3/`. A valid completed cache is reused on later runs. This cell does not fit either model.


In [ ]:
prepared_features = prepare_clean_slate_screen_features(
    root=REPO_DIR,
    output_root=LOCAL_TASK_DIR,
    local_work_dir=LOCAL_FEATURE_WORK_DIR,
)
print('Audit hash:', prepared_features['audit_contract']['audit_contract_hash'])
print('Gender feature cache:', prepared_features['gender'])
print('Usage feature cache:', prepared_features['usage'])


## 5. Train Gender Screen 1

For each outer fold, the small `C` grid and sigmoid calibration reuse the four remaining canonical folds as inner checks. No new split is created. Completed matching folds are reused after a disconnect.


In [ ]:
gender_anchor = (
    LOCAL_EVIDENCE_DIR
    / 'experiments/t3_gender_e9_semantic_filter/gender/aggregate/oof_predictions.csv'
)
gender_screen = run_clean_slate_gender_screen(
    prepared_features=prepared_features,
    output_root=LOCAL_TASK_DIR,
    folds=(0, 4),
    registry_path=LOCAL_REGISTRY,
    registry_mirrors=(),
    root=REPO_DIR,
    anchor_prediction_path=gender_anchor,
    reuse_completed=True,
)
print('Gender metrics:', gender_screen['metrics_path'])
print('Gender screen gate:', gender_screen['metrics']['screen_gate'])


## 6. Train Usage Screen 1

The article-type classifier and type-to-usage table are refitted independently inside each outer fold. Hyperparameter checks reuse the four remaining canonical folds, so no new split is created. No true validation article type is used to make a usage prediction. Completed matching folds are reused after a disconnect.


In [ ]:
usage_anchor = (
    LOCAL_EVIDENCE_DIR
    / 'experiments/t3_usage_e8_translation/usage/aggregate/oof_predictions.csv'
)
usage_screen = run_clean_slate_usage_screen(
    prepared_features=prepared_features,
    output_root=LOCAL_TASK_DIR,
    folds=(0, 4),
    registry_path=LOCAL_REGISTRY,
    registry_mirrors=(),
    root=REPO_DIR,
    anchor_prediction_path=usage_anchor,
    reuse_completed=True,
)
print('Usage metrics:', usage_screen['metrics_path'])
print('Usage screen gate:', usage_screen['metrics']['screen_gate'])


## 7. Two-fold screen summary

These values cover folds 0 and 4 only. They must not be compared with a five-fold aggregate. The saved gate uses matched rows from the downloaded historical evidence.


In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        'target': result['target'],
        'model_family': result['metrics']['model_family'],
        'folds': result['metrics']['validation_folds'],
        'macro_f1': result['metrics']['macro_f1'],
        'fold_sd': result['metrics']['fold_macro_f1_sample_sd'],
        'screen_gate': result['metrics']['screen_gate']['status'],
    }
    for result in (gender_screen, usage_screen)
])
display(summary)
print('Stop here. Review the saved evidence before advancing either model to five folds.')
